
Date: 2026-09-23
Tags: [[Programming language]], [[Python]], [[Algorithm]], [[Data Structure]], [[Array]]

---

# Dynamic Arrays & Amortized Analysis

> [!info] Definitions & Properties 
> A dynamic array resolves the limitation of fixed-capacity hardware arrays by allocating a larger array and migrating existing elements when the current capacity is exhausted. Python’s `list` class relies on dynamic array abstraction. Amortized analysis proves that occasional expensive reallocation operations are offset by a sequence of cheap insertions, yielding $O(1)$ average time per operation.
> 
>   

In [ ]:
import sys

data = []
for k in range(n):
	a = len(data)
	b = sys.getsizeof(data)
	print(f'Length: {a}, Size in Bytes: {}')
	data.append(None)

## 1. Dynamic Array Mechanics

When creating a low-level array, the precise size must be declared to allocate consecutive memory. If capacity is exhausted during an `append`, the dynamic array executes the following reallocation protocol:

1. Allocate a new array $B$ with larger capacity.    
    
2. Set $B[i] = A[i]$ for $i = 0, \dots, n-1$, where $n$ is the current number of items.
    
3. Set $A = B$, replacing the old array reference.      
    
4. Insert the new element into the new array.

> [!NOTE] Figure 5.12
> 
>   
> ![[Pasted image 20260923230932.png]]
> 
>   

### Python `DynamicArray` Implementation

In [ ]:
import ctypes

class DynamicArray:
    """A dynamic array class akin to a simplified Python list."""
    
    def __init__(self):
        """Create an empty array."""
        self._n = 0                                    # count actual elements
        self._capacity = 1                             # default array capacity
        self._A = self._make_array(self._capacity)     # low-level array
        
    def __len__(self):
        """Return number of elements stored in the array."""
        return self._n
        
    def __getitem__(self, k):
        """Return element at index k."""
        if not 0 <= k < self._n:
            raise IndexError('invalid index')
        return self._A[k]                              # retrieve from array
        
    def append(self, obj):
        """Add object to end of the array."""
        if self._n == self._capacity:                  # not enough room
            self._resize(2 * self._capacity)           # so double capacity
        self._A[self._n] = obj
        self._n += 1
        
    def insert(self, k , value):
	    if self._n == self._capacity:
		    self._resize(2 * self._capacity)
		for j in range(self._n, k, -1):
			self._A[j] = self._A[j-1]
		self._A[k] = value
		self._n += 1
		
	def remove(self, value):
		''' Remove first occurence of value (or raise ValueError)'''
		for k in range(self._n):
			if self._A[k] == value:
				for j in range(k, self._n-1):
					self._A[j] = self._A[j+1]
				self._A[self._n-1] = None
				self._n -= 1
				return
		return ValueError('value not found')	 
        
    def _resize(self, c):                              # nonpublic utility
        """Resize internal array to capacity c."""
        B = self._make_array(c)                        # new (bigger) array
        for k in range(self._n):                       # for each existing value
            B[k] = self._A[k]
        self._A = B                                    # use the bigger array
        self._capacity = c
        
    def _make_array(self, c):                          # nonpublic utility
        """Return new array with capacity c."""
        return (c * ctypes.py_object)()                # see ctypes documentation

## 2. Amortized Analysis of Appends

The strategy of geometric capacity expansion (e.g., doubling the capacity when full) guarantees that $n$ append operations execute in $O(n)$ total time, yielding an amortized $O(1)$ time per append.

> [!info] Proposition 5.1 (Geometric Expansion) 
> Let $S$ be a sequence implemented by means of a dynamic array with initial capacity one, using the strategy of doubling the array size when full. The total time to perform a series of $n$ append operations in $S$, starting from $S$ being empty, is $O(n)$.
> 
>   

### Proof by Accounting Method (Cyber-Dollars)

Assume 1 cyber-dollar pays for inserting one element.

- Charge each `append` operation 3 cyber-dollars.
    
- 1 dollar pays for the immediate insertion.
    
- 1 dollar is stored with the newly inserted element to pay for its future relocation.
    
- 1 dollar is stored with an older element (from the first half of the array) to pay for its future relocation.

When the array overflows at size $2^i$, there are exactly $2^{i-1}$ elements inserted since the last resize. These $2^{i-1}$ elements carry 2 dollars each, perfectly covering the cost to relocate all $2^i$ elements into the new array. Because the maximum charge per operation is constant ($3$), the total time for $n$ operations is bounded by $3n \implies O(n)$.


> [!NOTE] Figure 5.14
> 
>   
> ![[Pasted image 20260923231252.png]]
> 
>   

## 3. The Pitfall of Arithmetic Progression

Resizing the array by a fixed constant increment $c$ (e.g., adding 100 cells at a time) destroys the amortized $O(1)$ bound.

> [!info] Proposition 5.2 (Arithmetic Expansion) 
> Performing a series of $n$ append operations on an initially empty dynamic array using a fixed increment with each resize takes $\Omega(n^2)$ time.
> 
>   

### Justification

During a series of $n$ appends, resizing occurs at capacities $c, 2c, 3c, \dots, mc$ where $m = \lceil n/c \rceil$. The time spent copying elements during resize events forms an arithmetic summation (see Proposition 3.3):


$$\sum_{i=1}^{m} ci = c \sum_{i=1}^{m} i = c \frac{m(m+1)}{2} \ge c \frac{\left(\frac{n}{c}\right)\left(\frac{n}{c} + 1\right)}{2} \ge \frac{n^2}{2c}$$

Therefore, the total time is $\Omega(n^2)$.

> [!NOTE] Figure 5.15
> 
>   
> ![[Pasted image 20260923231314.png]]
> 
>   

## 4. Python's List Class Behavior

Python's built-in `list` class relies on dynamic arrays but employs an internal expansion formula rather than pure geometric doubling to balance memory usage and run-time efficiency.

Empirical testing confirms the $O(1)$ amortized cost:

In [ ]:
from time import time

def compute_average(n):
    """Perform n appends to an empty list and return average time elapsed."""
    data = []
    start = time()                             # record the start time (in seconds)
    for k in range(n):
        data.append(None)
    end = time()                               # record the end time (in seconds)
    return (end - start) / n                   # compute average per operation

### Table 5.2: Average running time of append

|**n**|**100**|**1,000**|**10,000**|**100,000**|**1,000,000**|**10,000,000**|**100,000,000**|
|---|---|---|---|---|---|---|---|
|**$\mu s$ per operation**|$0.219$|$0.158$|$0.164$|$0.151$|$0.147$|$0.147$|$0.149$|

> [!warning] Memory Usage and Shrinking 
> Repeatedly removing elements from a dynamic array can leave it mostly empty while retaining a large memory capacity. Robust implementations must occasionally shrink the underlying array while avoiding oscillation (e.g., shrinking only when capacity falls below $25\%$).

---
# ~={red}References=~
- [[Data_Structures_and_Algorithms_in_Python_Michael_T_Goodrich_z_library.pdf]](p, 192)